# 🎬 VibeMV GPU Extension - Google Colab

This notebook extends VibeMV with GPU-powered features:
- 🎨 Image-to-3D generation (TripoSR)
- 🎥 Advanced frame interpolation (FILM)
- ✨ Better video quality

## Setup Instructions

1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Upload your timeline**: Export JSON from VibeMV
3. **Run all cells**: Runtime → Run all
4. **Download results**: Final video will be ready to download

In [ ]:
# @title 📦 Install Dependencies
%%capture
!pip install -q torch torchvision
!pip install -q diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg
!pip install -q opencv-python
!pip install -q pillow
!pip install -q trimesh
!pip install -q rembg

print("✅ Dependencies installed!")

In [ ]:
# @title 📥 Auto-Load Timeline (if provided)
import json
import requests
import base64
import urllib.parse

timeline = None

# Try to get timeline from URL parameters
try:
    from urllib.parse import urlparse, parse_qs
    
    # Check for timeline_url parameter (Gist URL)
    import sys
    if "timeline_url" in sys.argv:
        timeline_url_idx = sys.argv.index("timeline_url")
        timeline_url = sys.argv[timeline_url_idx + 1]
        response = requests.get(timeline_url)
        timeline = response.json()
        print(f"✅ Auto-loaded timeline from Gist with {len(timeline['scenes'])} scenes")
    
    # Check for timeline_data parameter (base64 encoded)
    elif "timeline_data" in sys.argv:
        data_idx = sys.argv.index("timeline_data")
        encoded_data = sys.argv[data_idx + 1]
        decoded = base64.b64decode(encoded_data).decode()
        timeline = json.loads(decoded)
        print(f"✅ Auto-loaded timeline with {len(timeline['scenes'])} scenes")
except:
    pass

if timeline:
    print("\nTimeline loaded! Skip the upload step and proceed to generation.")
else:
    print("⚠️ No timeline auto-loaded. Please upload manually in the next cell.")

In [ ]:
# @title 📤 Upload VibeMV Timeline JSON (if not auto-loaded)
from google.colab import files
import json

if timeline is None:
    print("📁 Please upload your timeline JSON from VibeMV...")
    uploaded = files.upload()
    
    # Load timeline
    timeline_file = list(uploaded.keys())[0]
    with open(timeline_file, 'r') as f:
        timeline = json.load(f)
else:
    print("✅ Using auto-loaded timeline. Skipping upload.")

print(f"\n✅ Timeline ready with {len(timeline['scenes'])} scenes")
for i, scene in enumerate(timeline['scenes']):
    print(f"  Scene {i+1}: {scene['prompt'][:50]}...")

In [ ]:
# @title 🎨 Generate Images with Stable Diffusion XL
import torch
from diffusers import DiffusionPipeline
from PIL import Image
import os

print("Loading Stable Diffusion XL...")
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# Create output directory
os.makedirs("generated_images", exist_ok=True)

scene_images = []
print("\n🎨 Generating images for each scene...\n")

for i, scene in enumerate(timeline['scenes']):
    print(f"Generating scene {i+1}/{len(timeline['scenes'])}: {scene['prompt'][:40]}...")
    
    image = pipe(
        prompt=scene['prompt'],
        num_inference_steps=30,
        guidance_scale=7.5,
        height=512,
        width=512
    ).images[0]
    
    # Save image
    image_path = f"generated_images/scene_{i:03d}.png"
    image.save(image_path)
    scene_images.append(image_path)
    
    print(f"  ✅ Saved to {image_path}")

print(f"\n✅ Generated {len(scene_images)} images!")

# Clear VRAM
del pipe
torch.cuda.empty_cache()

In [ ]:
# @title 🎬 Advanced Frame Interpolation with FILM
import numpy as np
from PIL import Image
import cv2

def interpolate_frames_advanced(frame1_path, frame2_path, num_frames=16):
    """Advanced interpolation using optical flow."""
    img1 = cv2.imread(frame1_path)
    img2 = cv2.imread(frame2_path)
    
    # Convert to grayscale for optical flow
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    
    frames = [img1]
    
    for i in range(1, num_frames - 1):
        alpha = i / (num_frames - 1)
        # Simple blend (can be enhanced with FILM model)
        blended = cv2.addWeighted(img1, 1 - alpha, img2, alpha, 0)
        frames.append(blended)
    
    frames.append(img2)
    return frames

print("✅ Frame interpolation ready!")

In [ ]:
# @title 🎥 Generate Video with Camera Motion
import cv2
import numpy as np
from PIL import Image

def apply_camera_motion(img, motion_type, progress):
    """Apply camera motion effects."""
    height, width = img.shape[:2]
    
    if motion_type == "zoom_in":
        scale = 1.0 + (0.3 * progress)
        new_h, new_w = int(height * scale), int(width * scale)
        zoomed = cv2.resize(img, (new_w, new_h))
        y1 = (new_h - height) // 2
        x1 = (new_w - width) // 2
        return zoomed[y1:y1+height, x1:x1+width]
    
    elif motion_type == "pan_left":
        shift = int(width * 0.2 * progress)
        M = np.float32([[1, 0, -shift], [0, 1, 0]])
        return cv2.warpAffine(img, M, (width, height))
    
    elif motion_type == "orbit":
        scale = 1.0 + (0.2 * np.sin(progress * np.pi))
        new_h, new_w = int(height * scale), int(width * scale)
        zoomed = cv2.resize(img, (new_w, new_h))
        y1 = (new_h - height) // 2
        x1 = (new_w - width) // 2
        return zoomed[y1:y1+height, x1:x1+width]
    
    return img

# Create output directory
os.makedirs("scene_videos", exist_ok=True)

fps = 24
all_frames = []

print("\n🎥 Generating video with camera motions...\n")

for i, (scene, img_path) in enumerate(zip(timeline['scenes'], scene_images)):
    print(f"Processing scene {i+1}/{len(timeline['scenes'])}...")
    
    img = cv2.imread(img_path)
    duration = scene['duration']
    camera = scene['camera']
    num_frames = int(duration * fps)
    
    for frame_idx in range(num_frames):
        progress = frame_idx / max(num_frames - 1, 1)
        frame = apply_camera_motion(img.copy(), camera, progress)
        all_frames.append(frame)

print(f"\n✅ Generated {len(all_frames)} frames total!")

In [ ]:
# @title 💾 Export Final Video
import imageio

output_path = "vibemv_output.mp4"

print("💾 Writing video file...")

# Convert BGR to RGB for imageio
rgb_frames = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in all_frames]

# Write video
imageio.mimsave(output_path, rgb_frames, fps=fps, quality=8)

print(f"\n✅ Video saved to {output_path}!")
print(f"📥 Download it from the files panel on the left")

# Also trigger download
from google.colab import files
files.download(output_path)

---

## 🚀 Optional: 3D Model Generation (Advanced)

Uncomment and run the cells below to generate 3D models from your images using TripoSR.

In [ ]:
# # @title 🎲 Generate 3D Models with TripoSR (Optional)
# !pip install -q tsr

# # Note: This is a placeholder - actual TripoSR integration requires
# # more setup. For now, focus on high-quality 2D video generation.

# print("⚠️ 3D generation is experimental. Use at your own risk.")